In [ ]:
!pip -q install -U transformers accelerate sentencepiece safetensors huggingface-hub scikit-learn pandas numpy torch tqdm

In [ ]:
%pip install -q -U "transformers>=4.40.0" "accelerate>=0.32.0" sentencepiece tiktoken

In [ ]:
import importlib, sys

mods = ["numpy", "pandas", "scipy", "sklearn", "torch", "transformers", "accelerate"]

print("python:", sys.version)
for m in mods:
    try:
        mod = importlib.import_module(m)
        print(f"{m}: {getattr(mod, '__version__', 'OK')}")
    except Exception as e:
        print(f"FAILED on {m}: {type(e).__name__}: {e}")
        break

python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
numpy: 2.2.6
pandas: 3.0.1
scipy: 1.13.1
sklearn: 1.8.0
torch: 2.11.0+cu130
transformers: 5.4.0
accelerate: 1.13.0


In [ ]:
import sys, subprocess, pkgutil

def ensure_pkg(import_name, pip_name=None):
    if pkgutil.find_loader(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or import_name])

ensure_pkg("sentencepiece", "sentencepiece")
ensure_pkg("safetensors", "safetensors")

/tmp/ipykernel_135637/1023372169.py:4: DeprecationWarning: 'pkgutil.find_loader' is deprecated and slated for removal in Python 3.14; use importlib.util.find_spec() instead
  if pkgutil.find_loader(import_name) is None:


In [ ]:
!pip install sentencepiece tiktoken

In [ ]:
import os, gc, json, math, random
from copy import deepcopy

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss, roc_curve
from transformers import BertTokenizer, BertForSequenceClassification, BertConfig, DataCollatorWithPadding, get_linear_schedule_with_warmup
from tqdm.auto import tqdm

os.environ["TOKENIZERS_PARALLELISM"] = "false"

SEED = 123
MODEL_NAME = "ashleygong03/bamman-burns-latin-bert"
DATA_PATH = "../../data/processed_texts/600ad_unlemmatized"
RESULTS_PATH = "../../results/600ad"
OUT_DIR = os.path.join(RESULTS_PATH, "bert_outputs_unlemmatized_600ad")
CORPUS_FILE = os.path.join(DATA_PATH, "corpus_600ad_unlemmatized.csv")
LABELS_FILE = os.path.join(DATA_PATH, "corpus_600ad_unlemmatized_labeled.csv")

MAX_LENGTH = 512
CHUNK_STRIDE = 256  # overlap of 256 tokens between consecutive windows
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 2
LR = 2e-5
WEIGHT_DECAY = 0.01
N_EPOCHS = 4
WARMUP_RATIO = 0.1
PATIENCE = 1
MIN_DELTA = 0.0
OUTER_FOLDS = 5
VAL_FRAC = 0.1
LEARNING_CURVE_FRACTIONS = [0.10, 0.25, 0.50, 0.75, 1.00]
LEARNING_CURVE_REPS = 2
MIN_PER_CLASS_LC = 20
USE_FP16 = torch.cuda.is_available()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

os.makedirs(OUT_DIR, exist_ok=True)

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# ── Class weighting helper ─────────────────────────────────────────────────────

def compute_class_weights(y):
    """Compute inverse-frequency class weights: weight_i = n_samples / (n_classes * count_i).
    Returns a tensor of shape (2,) for [non_christian, christian]."""
    y = np.asarray(y, dtype=int)
    counts = np.bincount(y, minlength=2).astype(float)
    weights = len(y) / (len(counts) * counts)
    return torch.tensor(weights, dtype=torch.float32)

# ── Utility functions ──────────────────────────────────────────────────────────

def normalize_label(x):
    return str(x).strip().lower().replace("-", "_").replace(" ", "_")

def safe_prob(p, eps=1e-15):
    p = np.asarray(p, dtype=float)
    return np.clip(p, eps, 1 - eps)

def sigmoid(x):
    x = np.asarray(x, dtype=float)
    return 1 / (1 + np.exp(-x))

def choose_threshold_youden(y_true, prob):
    fpr, tpr, thr = roc_curve(y_true, prob)
    j = tpr - fpr
    idx = int(np.argmax(j))
    th = thr[idx]
    if not np.isfinite(th):
        return 0.5
    return float(np.clip(th, 0, 1))

def choose_threshold_f1(y_true, prob):
    y_true = np.asarray(y_true, dtype=int)
    prob = safe_prob(prob)
    cand = np.unique(np.concatenate([[0.0], prob, [0.5], [1.0]]))
    best_th, best_f1 = 0.5, -1.0
    for th in cand:
        pred = (prob >= th).astype(int)
        tp = ((pred == 1) & (y_true == 1)).sum()
        fp = ((pred == 1) & (y_true == 0)).sum()
        fn = ((pred == 0) & (y_true == 1)).sum()
        den = 2 * tp + fp + fn
        f1 = 0.0 if den == 0 else 2 * tp / den
        if f1 > best_f1:
            best_f1 = f1
            best_th = float(th)
    return best_th

def class_stats(y_true, prob, threshold):
    y_true = np.asarray(y_true, dtype=int)
    pred = (safe_prob(prob) >= threshold).astype(int)
    tp = ((pred == 1) & (y_true == 1)).sum()
    tn = ((pred == 0) & (y_true == 0)).sum()
    fp = ((pred == 1) & (y_true == 0)).sum()
    fn = ((pred == 0) & (y_true == 1)).sum()
    precision = np.nan if (tp + fp) == 0 else tp / (tp + fp)
    recall = np.nan if (tp + fn) == 0 else tp / (tp + fn)
    f1 = np.nan if (not np.isfinite(precision) or not np.isfinite(recall) or precision + recall == 0) else 2 * precision * recall / (precision + recall)
    accuracy = (tp + tn) / len(y_true)
    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}

def metric_row(y_true, prob, model_name="bert"):
    y_true = np.asarray(y_true, dtype=int)
    prob = safe_prob(prob)
    th_y = choose_threshold_youden(y_true, prob)
    th_f = choose_threshold_f1(y_true, prob)
    st_y = class_stats(y_true, prob, th_y)
    st_f = class_stats(y_true, prob, th_f)
    return {
        "model": model_name,
        "auc": roc_auc_score(y_true, prob),
        "log_loss": log_loss(y_true, prob, labels=[0, 1]),
        "brier": brier_score_loss(y_true, prob),
        "threshold_youden": th_y,
        "accuracy_youden": st_y["accuracy"],
        "precision_youden": st_y["precision"],
        "recall_youden": st_y["recall"],
        "f1_youden": st_y["f1"],
        "threshold_f1": th_f,
        "accuracy_f1": st_f["accuracy"],
        "precision_f1": st_f["precision"],
        "recall_f1": st_f["recall"],
        "f1_f1": st_f["f1"],
    }

# ── Dataset and data loading ──────────────────────────────────────────────────

class EncodedDataset(Dataset):
    """Encodes each text as one or more overlapping chunks of max_length tokens.
    Stores a doc_id per chunk so predictions can be aggregated back to documents."""
    def __init__(self, texts, labels, tokenizer, max_length, stride=CHUNK_STRIDE):
        self.input_ids = []
        self.attention_mask = []
        self.labels_list = []
        self.doc_ids = []

        for doc_idx, text in enumerate(texts):
            enc = tokenizer(
                text,
                truncation=False,
                padding=False,
                return_attention_mask=True,
            )
            ids = enc["input_ids"]
            mask = enc["attention_mask"]

            if len(ids) <= max_length:
                self.input_ids.append(ids[:max_length])
                self.attention_mask.append(mask[:max_length])
                self.doc_ids.append(doc_idx)
                if labels is not None:
                    self.labels_list.append(int(labels[doc_idx]))
            else:
                start = 0
                while start < len(ids):
                    end = min(start + max_length, len(ids))
                    self.input_ids.append(ids[start:end])
                    self.attention_mask.append(mask[start:end])
                    self.doc_ids.append(doc_idx)
                    if labels is not None:
                        self.labels_list.append(int(labels[doc_idx]))
                    if end == len(ids):
                        break
                    start += stride

        self.has_labels = labels is not None

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        item = {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
        }
        if self.has_labels:
            item["labels"] = self.labels_list[idx]
        return item

def make_loaders(tokenizer, train_texts, train_y, val_texts=None, val_y=None, batch_size_train=TRAIN_BATCH_SIZE, batch_size_eval=EVAL_BATCH_SIZE):
    collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8 if USE_FP16 else None)
    train_ds = EncodedDataset(train_texts, train_y, tokenizer, MAX_LENGTH)
    train_loader = DataLoader(train_ds, batch_size=batch_size_train, shuffle=True, collate_fn=collator, pin_memory=torch.cuda.is_available())
    val_loader = None
    val_doc_ids = None
    if val_texts is not None:
        val_ds = EncodedDataset(val_texts, val_y, tokenizer, MAX_LENGTH)
        val_doc_ids = val_ds.doc_ids
        val_loader = DataLoader(val_ds, batch_size=batch_size_eval, shuffle=False, collate_fn=collator, pin_memory=torch.cuda.is_available())
    return train_loader, val_loader, val_doc_ids

# ── Inference helpers ──────────────────────────────────────────────────────────

@torch.no_grad()
def predict_logits(model, loader):
    """Returns chunk-level logits (one row per chunk)."""
    model.eval()
    all_logits = []
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        batch.pop("labels", None)
        out = model(**batch)
        all_logits.append(out.logits.detach().cpu())
    return torch.cat(all_logits, dim=0).numpy()

def aggregate_chunk_logits(chunk_logits, doc_ids, n_docs):
    """Average chunk-level logits to produce one logit vector per document."""
    doc_logits = np.zeros((n_docs, chunk_logits.shape[1]), dtype=np.float64)
    doc_counts = np.zeros(n_docs, dtype=np.float64)
    for i, did in enumerate(doc_ids):
        doc_logits[did] += chunk_logits[i]
        doc_counts[did] += 1
    doc_counts[doc_counts == 0] = 1
    return doc_logits / doc_counts[:, None]

def fit_temperature(margins, y, lr=0.05, steps=300):
    margins = torch.tensor(np.asarray(margins), dtype=torch.float32, device=DEVICE)
    y = torch.tensor(np.asarray(y), dtype=torch.float32, device=DEVICE)
    log_t = torch.nn.Parameter(torch.zeros(1, device=DEVICE))
    opt = torch.optim.Adam([log_t], lr=lr)
    loss_fn = torch.nn.BCEWithLogitsLoss()
    best_loss = float("inf")
    best_t = 1.0
    for _ in range(steps):
        opt.zero_grad(set_to_none=True)
        t = torch.exp(log_t).clamp(1e-3, 1e3)
        loss = loss_fn(margins / t, y)
        loss.backward()
        opt.step()
        cur = float(loss.detach().cpu())
        if cur < best_loss:
            best_loss = cur
            best_t = float(t.detach().cpu().item())
    return best_t

def logits_to_margin(logits_2col):
    logits_2col = np.asarray(logits_2col)
    return logits_2col[:, 1] - logits_2col[:, 0]

def margins_to_prob(margins, temperature=1.0):
    return safe_prob(sigmoid(np.asarray(margins) / float(temperature)))

def margins_to_logit(margins, temperature=1.0):
    return np.asarray(margins) / float(temperature)

def make_train_val_split(y, seed, val_frac=VAL_FRAC):
    y = np.asarray(y, dtype=int)
    counts = np.bincount(y, minlength=2)
    if counts.min() < 2 or len(y) < 4:
        idx = np.arange(len(y))
        rng = np.random.default_rng(seed)
        rng.shuffle(idx)
        cut = max(1, min(len(y) - 1, int(round(len(y) * (1 - val_frac)))))
        return idx[:cut], idx[cut:]
    test_size = max(2, int(round(len(y) * val_frac)))
    test_size = min(test_size, len(y) - 2)
    sss = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    tr_idx, va_idx = next(sss.split(np.zeros(len(y)), y))
    return tr_idx, va_idx

# ── Model building and training ───────────────────────────────────────────────

def build_model_and_tokenizer():
    tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

    added_pad = False
    if tokenizer.pad_token is None:
        if tokenizer.eos_token is not None:
            tokenizer.pad_token = tokenizer.eos_token
        elif tokenizer.sep_token is not None:
            tokenizer.pad_token = tokenizer.sep_token
        else:
            tokenizer.add_special_tokens({"pad_token": "<pad>"})
            added_pad = True

    config = BertConfig.from_pretrained(MODEL_NAME)
    config.num_labels = 2
    model = BertForSequenceClassification.from_pretrained(MODEL_NAME, config=config)

    if added_pad:
        model.resize_token_embeddings(len(tokenizer))
    model.config.pad_token_id = tokenizer.pad_token_id
    model.to(DEVICE)
    return tokenizer, model

def train_one_model(train_texts, train_y, val_texts, val_y, seed):
    set_seed(seed)
    tokenizer, model = build_model_and_tokenizer()
    train_loader, val_loader, val_doc_ids = make_loaders(tokenizer, train_texts, train_y, val_texts, val_y)

    # Compute class weights from the training labels and create weighted loss
    class_weights = compute_class_weights(train_y).to(DEVICE)
    loss_fn = nn.CrossEntropyLoss(weight=class_weights)
    print(f"  class weights: non_christian={class_weights[0]:.3f}, christian={class_weights[1]:.3f}")

    n_steps = max(1, math.ceil(len(train_loader) / GRAD_ACCUM_STEPS) * N_EPOCHS)
    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(WARMUP_RATIO * n_steps),
        num_training_steps=n_steps
    )
    scaler = torch.cuda.amp.GradScaler(enabled=USE_FP16)

    best_state = None
    best_val_loss = float("inf")
    bad_epochs = 0

    for epoch in range(N_EPOCHS):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running_loss = 0.0

        pbar = tqdm(train_loader, desc=f"train epoch {epoch + 1}/{N_EPOCHS}", leave=False)
        for step, batch in enumerate(pbar, start=1):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            labels = batch.pop("labels")

            with torch.cuda.amp.autocast(enabled=USE_FP16):
                out = model(**batch)
                # Use our weighted loss instead of the model's built-in loss
                loss = loss_fn(out.logits, labels) / GRAD_ACCUM_STEPS

            scaler.scale(loss).backward()
            running_loss += float(loss.detach().cpu()) * GRAD_ACCUM_STEPS

            if step % GRAD_ACCUM_STEPS == 0 or step == len(train_loader):
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()

            pbar.set_postfix(train_loss=running_loss / step)

        val_chunk_logits = predict_logits(model, val_loader)
        val_logits = aggregate_chunk_logits(val_chunk_logits, val_doc_ids, len(val_y))
        val_margin = logits_to_margin(val_logits)
        val_prob = margins_to_prob(val_margin, temperature=1.0)
        val_loss = log_loss(val_y, val_prob, labels=[0, 1])

        print(f"epoch {epoch + 1}: val_log_loss={val_loss:.5f}")

        if val_loss + MIN_DELTA < best_val_loss:
            best_val_loss = val_loss
            best_state = deepcopy(model.state_dict())
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs > PATIENCE:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    val_chunk_logits = predict_logits(model, val_loader)
    val_logits = aggregate_chunk_logits(val_chunk_logits, val_doc_ids, len(val_y))
    val_margin = logits_to_margin(val_logits)
    temperature = fit_temperature(val_margin, val_y)

    return tokenizer, model, temperature, float(best_val_loss)

def predict_texts(model, tokenizer, texts, temperature):
    collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8 if USE_FP16 else None)
    ds = EncodedDataset(texts, labels=None, tokenizer=tokenizer, max_length=MAX_LENGTH)
    doc_ids = ds.doc_ids
    loader = DataLoader(ds, batch_size=EVAL_BATCH_SIZE, shuffle=False, collate_fn=collator, pin_memory=torch.cuda.is_available())
    chunk_logits = predict_logits(model, loader)
    logits = aggregate_chunk_logits(chunk_logits, doc_ids, len(texts))
    margin_raw = logits_to_margin(logits)
    logit_cal = margins_to_logit(margin_raw, temperature)
    prob_cal = margins_to_prob(margin_raw, temperature)
    return {"margin_raw": margin_raw, "logit_cal": logit_cal, "prob_cal": prob_cal}

def cleanup(model=None):
    if model is not None:
        del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ── Load data ──────────────────────────────────────────────────────────────────

df = pd.read_csv(CORPUS_FILE)
df["id"] = df["id"].astype(str)
df = df.loc[df["text"].notna() & (df["text"].astype(str).str.len() > 0)].copy()
df = df.reset_index(drop=True)
df["row_id"] = np.arange(1, len(df) + 1)

dup_ids = df["id"].value_counts()
dup_ids = dup_ids[dup_ids > 1]
if len(dup_ids):
    raise ValueError(f"Duplicate ids in corpus: {dup_ids.index[:10].tolist()}")

labels_raw = pd.read_csv(LABELS_FILE)
labels_raw["id"] = labels_raw["id"].astype(str)
label_col = "manual_label" if "manual_label" in labels_raw.columns else "label"
labels_raw["manual_label"] = labels_raw[label_col].map(normalize_label)
labels_raw = labels_raw.loc[labels_raw["manual_label"].isin(["christian", "non_christian"]), ["id", "manual_label"]].copy()

conflicts = labels_raw.groupby("id")["manual_label"].nunique()
conflicts = conflicts[conflicts > 1]
if len(conflicts):
    raise ValueError(f"Conflicting labels: {conflicts.index[:10].tolist()}")

labels = labels_raw.drop_duplicates("id").copy()
label_map = {"non_christian": 0, "christian": 1}
labels["y"] = labels["manual_label"].map(label_map).astype(int)

lab = df.merge(labels, on="id", how="inner").sort_values("row_id").reset_index(drop=True)
if lab["y"].nunique() != 2:
    raise ValueError("Need both classes in labeled data.")

# Print class distribution
print("Class distribution:")
print(lab["manual_label"].value_counts())
print(f"Class ratio: {lab['y'].mean():.1%} christian, {1 - lab['y'].mean():.1%} non_christian")
print(f"Class weights that will be used: {compute_class_weights(lab['y'].values)}")
print()

# ── Out-of-fold cross-validation ──────────────────────────────────────────────

bert_oof_rows = []
oof_prob = np.full(len(lab), np.nan)
oof_logit = np.full(len(lab), np.nan)
oof_margin = np.full(len(lab), np.nan)
oof_fold = np.full(len(lab), np.nan)

n_outer = min(OUTER_FOLDS, lab["y"].value_counts().min())
if n_outer < 2:
    raise ValueError("Need at least 2 examples in the minority class for stratified OOF training.")

skf = StratifiedKFold(n_splits=n_outer, shuffle=True, random_state=SEED)

for fold, (tr_idx, te_idx) in enumerate(skf.split(lab["text"], lab["y"]), start=1):
    print(f"\nouter fold {fold}/{skf.get_n_splits()}")
    train_df = lab.iloc[tr_idx].reset_index(drop=True)
    test_df = lab.iloc[te_idx].reset_index(drop=True)

    tr2, va2 = make_train_val_split(train_df["y"].values, seed=SEED + fold, val_frac=VAL_FRAC)
    inner_train = train_df.iloc[tr2].reset_index(drop=True)
    inner_val = train_df.iloc[va2].reset_index(drop=True)

    tokenizer, model, temperature, best_val_loss = train_one_model(
        inner_train["text"].tolist(),
        inner_train["y"].tolist(),
        inner_val["text"].tolist(),
        inner_val["y"].tolist(),
        seed=SEED + fold
    )

    pred = predict_texts(model, tokenizer, test_df["text"].tolist(), temperature)

    oof_prob[te_idx] = pred["prob_cal"]
    oof_logit[te_idx] = pred["logit_cal"]
    oof_margin[te_idx] = pred["margin_raw"]
    oof_fold[te_idx] = fold

    fold_rows = pd.DataFrame({
        "row_id": test_df["row_id"].values,
        "id": test_df["id"].values,
        "manual_label": test_df["manual_label"].values,
        "fold": fold,
        "prob_bert": pred["prob_cal"],
        "score_bert_logit": pred["logit_cal"],
        "score_bert_margin": pred["margin_raw"],
        "temperature": temperature,
        "val_log_loss": best_val_loss
    })
    bert_oof_rows.append(fold_rows)

    cleanup(model)

bert_oof = pd.concat(bert_oof_rows, ignore_index=True).sort_values("row_id").reset_index(drop=True)
oof_metrics = pd.DataFrame([metric_row(lab["y"].values, oof_prob, "bert")])
threshold_youden = float(oof_metrics.loc[0, "threshold_youden"])
threshold_f1 = float(oof_metrics.loc[0, "threshold_f1"])

bert_oof["threshold_youden"] = threshold_youden
bert_oof["threshold_f1"] = threshold_f1
bert_oof["label_bert_youden"] = np.where(bert_oof["prob_bert"].values >= threshold_youden, "christian", "non_christian")
bert_oof["label_bert_f1"] = np.where(bert_oof["prob_bert"].values >= threshold_f1, "christian", "non_christian")

# ── Full model training ───────────────────────────────────────────────────────

full_tr, full_va = make_train_val_split(lab["y"].values, seed=SEED + 999, val_frac=VAL_FRAC)
full_train = lab.iloc[full_tr].reset_index(drop=True)
full_val = lab.iloc[full_va].reset_index(drop=True)

tokenizer, model, temperature_full, best_val_loss_full = train_one_model(
    full_train["text"].tolist(),
    full_train["y"].tolist(),
    full_val["text"].tolist(),
    full_val["y"].tolist(),
    seed=SEED + 999
)

full_pred = predict_texts(model, tokenizer, df["text"].tolist(), temperature_full)

bert_full = df[["row_id", "id"]].copy()
bert_full["prob_bert"] = full_pred["prob_cal"]
bert_full["score_bert_logit"] = full_pred["logit_cal"]
bert_full["score_bert_margin"] = full_pred["margin_raw"]
bert_full["threshold_youden"] = threshold_youden
bert_full["threshold_f1"] = threshold_f1
bert_full["label_bert_youden"] = np.where(bert_full["prob_bert"].values >= threshold_youden, "christian", "non_christian")
bert_full["label_bert_f1"] = np.where(bert_full["prob_bert"].values >= threshold_f1, "christian", "non_christian")
bert_full["temperature"] = temperature_full
bert_full["val_log_loss"] = best_val_loss_full

cleanup(model)

# ── Learning curve ─────────────────────────────────────────────────────────────

def stratified_subset_indices(y, frac, seed, min_per_class=20):
    y = np.asarray(y, dtype=int)
    idx_pos = np.where(y == 1)[0]
    idx_neg = np.where(y == 0)[0]
    rng = np.random.default_rng(seed)
    n_pos = min(len(idx_pos), max(min_per_class, int(np.floor(len(idx_pos) * frac))))
    n_neg = min(len(idx_neg), max(min_per_class, int(np.floor(len(idx_neg) * frac))))
    keep = np.concatenate([
        rng.choice(idx_pos, size=n_pos, replace=False),
        rng.choice(idx_neg, size=n_neg, replace=False)
    ])
    rng.shuffle(keep)
    return keep

lc_rows = []

for frac in LEARNING_CURVE_FRACTIONS:
    for rep in range(1, LEARNING_CURVE_REPS + 1):
        print(f"\nlearning curve frac={frac:.2f} rep={rep}")
        idx = stratified_subset_indices(lab["y"].values, frac=frac, seed=7000 + rep, min_per_class=MIN_PER_CLASS_LC)
        sub = lab.iloc[idx].reset_index(drop=True)

        sub_oof_prob = np.full(len(sub), np.nan)
        n_sub_outer = min(OUTER_FOLDS, sub["y"].value_counts().min())
        sub_skf = StratifiedKFold(n_splits=n_sub_outer, shuffle=True, random_state=9000 + rep)

        for fold, (tr_idx, te_idx) in enumerate(sub_skf.split(sub["text"], sub["y"]), start=1):
            print(f"  fold {fold}/{sub_skf.get_n_splits()}")
            train_df = sub.iloc[tr_idx].reset_index(drop=True)
            test_df = sub.iloc[te_idx].reset_index(drop=True)

            tr2, va2 = make_train_val_split(train_df["y"].values, seed=11000 + rep + fold, val_frac=VAL_FRAC)
            inner_train = train_df.iloc[tr2].reset_index(drop=True)
            inner_val = train_df.iloc[va2].reset_index(drop=True)

            tokenizer, model, temperature, _ = train_one_model(
                inner_train["text"].tolist(),
                inner_train["y"].tolist(),
                inner_val["text"].tolist(),
                inner_val["y"].tolist(),
                seed=12000 + rep + fold
            )

            pred = predict_texts(model, tokenizer, test_df["text"].tolist(), temperature)
            sub_oof_prob[te_idx] = pred["prob_cal"]

            cleanup(model)

        m = metric_row(sub["y"].values, sub_oof_prob, "bert")
        lc_rows.append({
            "frac": frac,
            "rep": rep,
            "auc": m["auc"],
            "log_loss": m["log_loss"],
            "brier": m["brier"],
            "threshold_youden": m["threshold_youden"],
            "threshold_f1": m["threshold_f1"],
            "f1_youden": m["f1_youden"],
            "f1_f1": m["f1_f1"]
        })

bert_learning_curve = pd.DataFrame(lc_rows)

# ── Save results ───────────────────────────────────────────────────────────────

bert_oof_file = os.path.join(OUT_DIR, "bert_oof_predictions.csv")
bert_full_file = os.path.join(OUT_DIR, "bert_full_predictions.csv")
bert_lc_file = os.path.join(OUT_DIR, "bert_learning_curve.csv")
bert_metrics_file = os.path.join(OUT_DIR, "bert_metrics_oof.csv")
bert_config_file = os.path.join(OUT_DIR, "bert_run_config.json")

bert_oof.to_csv(bert_oof_file, index=False)
bert_full.to_csv(bert_full_file, index=False)
bert_learning_curve.to_csv(bert_lc_file, index=False)
oof_metrics.to_csv(bert_metrics_file, index=False)

with open(bert_config_file, "w") as f:
    json.dump({
        "model_name": MODEL_NAME,
        "max_length": MAX_LENGTH,
        "train_batch_size": TRAIN_BATCH_SIZE,
        "eval_batch_size": EVAL_BATCH_SIZE,
        "grad_accum_steps": GRAD_ACCUM_STEPS,
        "lr": LR,
        "weight_decay": WEIGHT_DECAY,
        "n_epochs": N_EPOCHS,
        "warmup_ratio": WARMUP_RATIO,
        "patience": PATIENCE,
        "outer_folds": OUTER_FOLDS,
        "val_frac": VAL_FRAC,
        "threshold_youden": threshold_youden,
        "threshold_f1": threshold_f1,
        "temperature_full": temperature_full
    }, f, indent=2)

print("\nSaved:")
print(bert_oof_file)
print(bert_full_file)
print(bert_lc_file)
print(bert_metrics_file)
print(bert_config_file)

Class distribution:
manual_label
non_christian    286
christian         76
Name: count, dtype: int64
Class ratio: 21.0% christian, 79.0% non_christian
Class weights that will be used: tensor([0.6329, 2.3816])


outer fold 1/5


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ashleygong03/bamman-burns-latin-bert
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  class weights: non_christian=0.634, christian=2.364


/var/folders/bt/x42yq40j415fmgs3cd9vkv9m0000gn/T/ipykernel_1567/4115426918.py:318: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_FP16)


train epoch 1/4:   0%|          | 0/1063 [00:00<?, ?it/s]

/var/folders/bt/x42yq40j415fmgs3cd9vkv9m0000gn/T/ipykernel_1567/4115426918.py:334: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_FP16):
